Read the [accompanying blog post](https://acytoo.com/ladder/llm-tut-2-musicgen/) for explanations and experiment results.

# prepare data

In [ ]:
# Download and extract both composers without an interactive overwrite prompt.
from pathlib import Path
import urllib.request
import zipfile

archive_path = Path("Piano_Two_Moods_12_MIDI.zip")
urllib.request.urlretrieve(
    "https://dl.acytoo.com/Piano_Two_Moods_12_MIDI.zip",
    archive_path,
)
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(".")
print("Ready: 01_slow_melancholic_satie / 02_bright_ragtime_joplin")


Ready: 01_slow_melancholic_satie / 02_bright_ragtime_joplin


In [ ]:
# Change only this parameter after downloading the data.
COMPOSER = "both"  # "satie", "joplin", or "both"

!pip install -q pretty_midi
import glob
import pretty_midi
import torch
import io
import mido
# Vocabulary (vocab_size = 308):
# 0..87:    NOTE_ON (piano 88 keys, MIDI pitch 21..108)
# 88..175:  NOTE_OFF (MIDI pitch 21..108)
# 176..275: TIME_SHIFT (1..100 steps, 10ms per step)
# 276..307: VELOCITY (32 bins covering 0..127)
vocab_size = 308
PITCH_MIN, PITCH_MAX = 21, 108
def encode_midi(midi_path: str) -> list[int]:
    # Merge tracks so tempo events on any track are honored by PrettyMIDI.
    midi = mido.MidiFile(midi_path)
    if midi.type == 2:
        raise ValueError(f"Asynchronous type-2 MIDI is unsupported: {midi_path}")
    midi.tracks = [mido.merge_tracks(midi.tracks)]
    midi.type = 0
    midi_buffer = io.BytesIO()
    midi.save(file=midi_buffer)
    midi_buffer.seek(0)
    pm = pretty_midi.PrettyMIDI(midi_buffer)
    events = []
    for inst in pm.instruments:
        if not inst.is_drum:
            for note in inst.notes:
                pitch = max(PITCH_MIN, min(PITCH_MAX, note.pitch)) - PITCH_MIN
                events.append((note.start, 1, pitch, note.velocity))  # 1 = ON
                events.append((note.end,   0, pitch, 0))              # 0 = OFF
    events.sort(key=lambda x: (x[0], x[1]))
    tokens = []
    current_time = 0.0
    current_vel_bin = -1
    for time, etype, pitch, vel in events:
        dt_steps = int(round((time - current_time) * 100))
        while dt_steps > 0:
            step = min(dt_steps, 100)
            tokens.append(176 + (step - 1))
            dt_steps -= step
        current_time = time
        if etype == 1:
            vel_bin = vel // 4
            if vel_bin != current_vel_bin:
                tokens.append(276 + vel_bin)
                current_vel_bin = vel_bin
            tokens.append(pitch)
        else:
            tokens.append(88 + pitch)
    return tokens
def decode_midi(tokens: list[int], out_path: str = "generated.mid"):
    pm = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)
    current_time = 0.0
    current_vel = 80
    active_notes = {}
    for tok in tokens:
        if 0 <= tok < 88:
            pitch = tok + PITCH_MIN
            active_notes[pitch] = (current_time, current_vel)
        elif 88 <= tok < 176:
            pitch = (tok - 88) + PITCH_MIN
            if pitch in active_notes:
                start, vel = active_notes.pop(pitch)
                if current_time > start:
                    piano.notes.append(pretty_midi.Note(
                        velocity=vel, pitch=pitch, start=start, end=current_time
                    ))
        elif 176 <= tok < 276:
            step = (tok - 176) + 1
            current_time += step * 0.01
        elif 276 <= tok < 308:
            current_vel = (tok - 276) * 4 + 2
    pm.instruments.append(piano)
    pm.write(out_path)
    return pm


from pathlib import Path

COMPOSER_DIRS = {
    "satie": Path("01_slow_melancholic_satie"),
    "joplin": Path("02_bright_ragtime_joplin"),
}
COMPOSER = COMPOSER.strip().lower()
if COMPOSER not in (*COMPOSER_DIRS, "both"):
    raise ValueError('COMPOSER must be "satie", "joplin", or "both"')
composer_names = tuple(COMPOSER_DIRS) if COMPOSER == "both" else (COMPOSER,)

composer_data = {}
composer_files = {}
for composer in composer_names:
    midi_files = sorted(
        p for p in COMPOSER_DIRS[composer].rglob("*")
        if p.suffix.lower() in {".mid", ".midi"}
    )
    if not midi_files:
        raise FileNotFoundError(
            f"No MIDI files for {composer}: {COMPOSER_DIRS[composer].resolve()}. "
            "Run the download/unzip cell first."
        )
    all_tokens = []
    for midi_file in midi_files:
        piece_tokens = encode_midi(str(midi_file))
        if not piece_tokens:
            raise ValueError(f"No playable notes in {midi_file}")
        all_tokens.extend(piece_tokens)
        all_tokens.extend([275, 275])  # 2 seconds between pieces
    composer_data[composer] = torch.tensor(all_tokens, dtype=torch.long)
    composer_files[composer] = [str(p) for p in midi_files]
    print(f"{composer}: {len(midi_files)} files | {len(all_tokens):,} tokens")
# Keep the existing batching demonstration usable with the first selected composer.
data = composer_data[composer_names[0]]


satie: 6 files | 8,211 tokens
joplin: 6 files | 41,524 tokens


In [ ]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
batch_size = 64
block_size =256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2


In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(5):
    context = x[:t + 1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([298]) the target: 20
when input is tensor([298,  20]) the target: 275
when input is tensor([298,  20, 275]) the target: 51
when input is tensor([298,  20, 275,  51]) the target: 44
when input is tensor([298,  20, 275,  51,  44]) the target: 39


In [ ]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1: i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('-----------------')
for b in range(5):
    for t in range(5):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([64, 256])
tensor([[124, 127, 136,  ..., 298,  22, 275],
        [ 43, 250, 112,  ...,  38, 250, 107],
        [138, 184, 107,  ..., 148, 133, 110],
        ...,
        [ 20, 275, 132,  ...,  44,  35, 275],
        [184,  57, 241,  ..., 141, 133, 110],
        [219, 134, 181,  ..., 127, 123, 200]], device='cuda:0')
targets:
torch.Size([64, 256])
tensor([[127, 136, 132,  ...,  22, 275, 143],
        [250, 112, 231,  ..., 250, 107, 231],
        [184, 107,  50,  ..., 133, 110, 293],
        ...,
        [275, 132,  39,  ...,  35, 275, 250],
        [ 57, 241, 145,  ..., 133, 110, 296],
        [134, 181, 115,  ..., 123, 200, 132]], device='cuda:0')
-----------------
when input is [124] the target: 127
when input is [124, 127] the target: 136
when input is [124, 127, 136] the target: 132
when input is [124, 127, 136, 132] the target: 105
when input is [124, 127, 136, 132, 105] the target: 299
when input is [43] the target: 250
when input is [43, 250] the target: 112
wh

# network

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)


In [ ]:
class Head(nn.Module):
    """one head of self attention"""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x) # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5 #(B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) #(B, T, head_size)
        out = wei @ v # (B, T, head_size)
        return out


class MultiHeadAttention(nn.Module):
    """Multi heads attention"""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # (B, T, num_heads * head_size)
        out = self.dropout(self.proj(out)) # (B, T, C)
        return out


class FeedForward(nn.Module):
    """a simple linear layer followed by a non-linearity"""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block: communication followed by computation"""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class SimpleLLM(nn.Module):
    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx) # B, T, C
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # B, T, C
        x = tok_emb + pos_emb # (B, T, C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size token
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T + 1)
        return idx

## Slow Model

In [ ]:
# Use only satie's pieces for this model.
slow_data = composer_data["satie"]
slow_n = int(0.9 * len(slow_data))
slow_train_data = slow_data[:slow_n]
slow_val_data = slow_data[slow_n:]

def get_slow_batch(split):
    data = slow_train_data if split == 'train' else slow_val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

slow = SimpleLLM().to(device)
slow_optimizer = torch.optim.AdamW(slow.parameters(), lr=1e-3)

### Train

In [ ]:
for slow_step in range(1000):
    # sample a batch of data
    slow_xb, slow_yb = get_slow_batch('train')

    # evaluate the loss
    slow_logits, slow_loss = slow(slow_xb, slow_yb)
    slow_optimizer.zero_grad(set_to_none=True)
    slow_loss.backward()
    slow_optimizer.step()

print(slow_loss.item())
torch.save(slow.state_dict(), "satie.pt")

0.03374803438782692


In [ ]:
!apt-get install -qq fluidsynth
!pip install -q pyfluidsynth
from IPython.display import Audio, display
import importlib
import pretty_midi.fluidsynth
importlib.reload(pretty_midi.fluidsynth)  # Refresh _HAS_FLUIDSYNTH flag without restarting runtime

Extracting templates from packages: 100%
Selecting previously unselected package libevdev2:amd64.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../00-libevdev2_1.13.1+dfsg-1build1_amd64.deb ...
Unpacking libevdev2:amd64 (1.13.1+dfsg-1build1) ...
Selecting previously unselected package at-spi2-common.
Preparing to unpack .../01-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../02-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../03-libxtst6_2%3a1.2.3-1.1ubuntu0.1_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1.1ubuntu0.1) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../04-session-migration_0.3.9build1_amd64.deb ...
Unpacking session-migration (0.3.

<module 'pretty_midi.fluidsynth' from '/usr/local/lib/python3.13/dist-packages/pretty_midi/fluidsynth.py'>

### Sampling

In [ ]:
slow.eval()
with torch.no_grad():
    # Start on C4 (MIDI pitch 60 -> token 39)
    slow_start_token = torch.tensor(
        [[60 - PITCH_MIN]], dtype=torch.long, device=device)
    slow_gen_tokens = slow.generate(
        slow_start_token, max_new_tokens=1000)[0].tolist()
slow.train();

slow_pm_gen = decode_midi(slow_gen_tokens, out_path="satie_gen.mid")
slow_audio_wave = slow_pm_gen.fluidsynth(fs=44100)
display(Audio(slow_audio_wave, rate=44100))

## Bright Model

In [ ]:
# Use only joplin's pieces for this model.
bright_data = composer_data["joplin"]
bright_n = int(0.9 * len(bright_data))
bright_train_data = bright_data[:bright_n]
bright_val_data = bright_data[bright_n:]

def get_bright_batch(split):
    data = bright_train_data if split == 'train' else bright_val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

bright = SimpleLLM().to(device)
bright_optimizer = torch.optim.AdamW(bright.parameters(), lr=1e-3)

In [ ]:
for bright_step in range(1000):
    # sample a batch of data
    bright_xb, bright_yb = get_bright_batch('train')

    # evaluate the loss
    bright_logits, bright_loss = bright(bright_xb, bright_yb)
    bright_optimizer.zero_grad(set_to_none=True)
    bright_loss.backward()
    bright_optimizer.step()

print(bright_loss.item())
torch.save(bright.state_dict(), "joplin.pt")

0.0537199005484581


In [ ]:
bright.eval()
with torch.no_grad():
    # Start on C4 (MIDI pitch 60 -> token 39)
    bright_start_token = torch.tensor(
        [[60 - PITCH_MIN]], dtype=torch.long, device=device)
    bright_gen_tokens = bright.generate(
        bright_start_token, max_new_tokens=1000)[0].tolist()
bright.train();

bright_pm_gen = decode_midi(bright_gen_tokens, out_path="joplin_gen.mid")
bright_audio_wave = bright_pm_gen.fluidsynth(fs=44100)
display(Audio(bright_audio_wave, rate=44100))

## Untrained Model

In [ ]:
# Randomly initialized model; no optimizer or training steps.
untrained = SimpleLLM().to(device)

In [ ]:
untrained.eval()
with torch.no_grad():
    # Start on C4 (MIDI pitch 60 -> token 39)
    untrained_start_token = torch.tensor(
        [[60 - PITCH_MIN]], dtype=torch.long, device=device)
    untrained_gen_tokens = untrained.generate(
        untrained_start_token, max_new_tokens=1000)[0].tolist()

untrained_pm_gen = decode_midi(untrained_gen_tokens, out_path="untrained_gen.mid")
untrained_audio_wave = untrained_pm_gen.fluidsynth(fs=44100)
display(Audio(untrained_audio_wave, rate=44100))